In [1]:
!pip install -q monai nibabel einops torch torchvision matplotlib tqdm

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from monai.networks.nets import SwinUNETR
from einops import rearrange
from scipy.stats import entropy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SwinUNETR(
    in_channels=3,
    out_channels=2,      
    feature_size=12,
    use_checkpoint=False
).to(device)

model.load_state_dict(
    torch.load("/Users/apple/Desktop/BRAINIAC/models/best_swin_unetr.pt",
               map_location=device)
)

model.eval()
print("✅ Model loaded successfully")

In [ ]:
attention_maps = []

def attention_hook(module, input, output):
    attention_maps.append(output.detach().cpu())

for stage in [
    model.swinViT.layers1,
    model.swinViT.layers2,
    model.swinViT.layers3,
    model.swinViT.layers4,
]:
    for layer in stage:
        for block in layer.blocks:
            block.attn.softmax.register_forward_hook(attention_hook)

print("✅ Hooks registered successfully")

In [ ]:
def load_sample(path):
    x = np.load(path)  
    x = torch.tensor(x).permute(3,0,1,2).unsqueeze(0).float()
    return x

x = load_sample("/Users/apple/Desktop/BRAINIAC/data/processed/images/X_148.npy").to(device)

In [ ]:
import torch
import torch.nn.functional as F

def compute_attention_entropy(attn_maps):
    entropies = []

    for attn in attn_maps:
        # Flatten spatial attention
        p = attn.flatten()
        
        # Normalize into probability distribution
        p = p / (p.sum() + 1e-8)
        
        # Compute entropy
        entropy = -torch.sum(p * torch.log(p + 1e-8))
        entropies.append(entropy)

    return torch.mean(torch.stack(entropies)).item()

In [ ]:
attention_maps.clear()

with torch.no_grad():
    y_hat = model(x)

print("Output shape:", y_hat.shape)
print("Collected attention maps:", len(attention_maps))

In [ ]:
def tokens_to_spatial(attn):
    """
    Convert Swin window attention → ONE spatial map per image.
    """

    if attn.ndim != 4:
        return None

    B, H, T, _ = attn.shape

    attn = attn.mean(dim=1)      
    attn = attn.mean(dim=-1)     

    window_tokens = 343

    if T % window_tokens != 0:
        return None

    num_windows = T // window_tokens

    attn = attn.view(B, num_windows, 7, 7, 7)

    
    attn = attn.mean(dim=1)   #7]

    attn = torch.nn.functional.interpolate(
        attn.unsqueeze(1),
        size=(128,128,128),
        mode="trilinear",
        align_corners=False
         ).squeeze(1)

    return attn

spatial_maps = []
for attn in attention_maps:
    m = tokens_to_spatial(attn)
    if m is not None:
        spatial_maps.append(m)

if len(spatial_maps) == 0:
    raise RuntimeError("❌ No valid attention maps captured — run inference again after registering hooks")

if len(spatial_maps) == 0:
    raise RuntimeError("❌ No valid attention maps — run inference again")

spatial_maps = [m for m in spatial_maps if m.shape == spatial_maps[0].shape]

final_attention = torch.mean(torch.stack(spatial_maps), dim=0)

attention_entropy = compute_attention_entropy([final_attention])
print("✅ Attention Entropy:", attention_entropy)


In [ ]:
slice_idx = 64

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.title("MRI Slice")
plt.imshow(x[0,0,:,:,slice_idx].cpu(), cmap="gray")
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Attention Map")
plt.imshow(final_attention[0,:,:,slice_idx].cpu(), cmap="hot")
plt.axis("off")

plt.show()